# Pipeline iPhone v1: CPU control saja

**GPU v1 diblokir setelah fatal assertion MuQ 3000-frame pada batch.** Lanjutkan dengan [encoder_glu_repair.ipynb](encoder_glu_repair.ipynb) untuk kandidat graph perbaikan. Notebook ini tetap tersedia untuk pembanding CPU; bukan jalur GPU lanjutan.


Validation 32 lagu / 451 section mempertahankan 100% label pipeline palette6 saat ini. Sekarang uji kedua encoder bersama pada iPhone, termasuk window akhir, pergantian shape dan head chunk, waktu awal, pengulangan dan memori.

Ini **project benchmark terpisah**. Tidak ada model aplikasi SongMap yang ditimpa atau routing produksi yang diaktifkan. Dua lagu validation dibundel otomatis: lagu pertama dan lagu lain berdurasi paling panjang, termasuk yang melewati 7 menit. Semua boundary, target manual dan checkpoint tetap.

Persiapan memeriksa seluruh rantai sumber, menjalankan typecheck Swift terhadap SDK iPhone sebelum menyalin aset besar, dan menjalankan pipeline PCM di CPU Mac untuk kedua lagu dengan **partial encoder asli secara nyata**. Mean audio, prior label dan konteks native harus sesuai dengan hasil validation. Tidak ada cache partial yang menggantikan inferensi dalam benchmark ini.

Project sementara memuat model dinamis dan statis terpisah sehingga bobot terduplikasi. Ini bukan ukuran paket produksi; deduplikasi/pengemasan dilakukan setelah jalur ini terbukti stabil.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib,json,subprocess

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/prepare_encoder_pipeline_iphone.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'  # Pertahankan path executable venv.
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
reports=[]
for p in (ROOT/'outputs.local').glob('encoder-pipeline-quality-*/validation-all.json'):
    r=json.loads(p.read_text())
    if (r.get('full_validation') and r.get('section_agreement')==1
            and r.get('comparison_to_current_pipeline',{}).get('no_observed_drop')):
        reports.append(p)
assert reports, 'Selesaikan encoder_pipeline_quality.ipynb sebelum menyiapkan benchmark ini'
QUALITY=max(reports,key=lambda p:p.stat().st_mtime).resolve()
OUTPUT=ROOT/'outputs.local'/('encoder-pipeline-phone-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
print('Quality gate:',QUALITY)
print('Output:',OUTPUT)

def prepare():
    if (OUTPUT/'preparation.json').is_file():
        saved=json.loads((OUTPUT/'preparation.json').read_text())
        assert saved['quality_sha256']==hashlib.sha256(QUALITY.read_bytes()).hexdigest()
        assert saved['mac_pcm_pipeline_passed'] and saved['native_typecheck_passed']
        sources={
            'preparation_script_sha256':ROOT/'scripts/prepare_encoder_pipeline_iphone.py',
            'template_runner_sha256':ROOT/'scripts/encoder_pipeline_iphone/BenchmarkRunner.swift',
            'template_view_sha256':ROOT/'scripts/encoder_pipeline_iphone/BenchmarkApp.swift',
            'app_math_sha256':ROOT/'SongMap/Data/SongAnalysis/Structure/SongFormerContextMath.swift',
            'app_bridge_sha256':ROOT/'SongMap/Data/SongAnalysis/Structure/SongFormerTensorBridge.swift',
        }
        for key,path in sources.items():
            assert saved[key]==hashlib.sha256(path.read_bytes()).hexdigest(), 'Sumber berubah; gunakan OUTPUT baru'
        assert saved['manifest_sha256']==hashlib.sha256((OUTPUT/'Assets/benchmark_manifest.json').read_bytes()).hexdigest()
        assert saved['runner_sha256']==hashlib.sha256((OUTPUT/'BenchmarkRunner.swift').read_bytes()).hexdigest()
        assert saved['view_sha256']==hashlib.sha256((OUTPUT/'BenchmarkApp.swift').read_bytes()).hexdigest()
        print('Project sudah siap:',OUTPUT/'Benchmark.xcodeproj')
        return
    command=[str(RUN_PYTHON),'-u',str(ROOT/'scripts/prepare_encoder_pipeline_iphone.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--songmap-root',str(ROOT),
        '--quality-report',str(QUALITY),'--output-dir',str(OUTPUT)]
    log=Path(str(OUTPUT)+'.prepare.log');log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Persiapan gagal ({status}); lihat {log}. Jangan lanjut build sebelum preflight selesai; pakai OUTPUT baru untuk percobaan ulang.')


In [ ]:
prepare()
proof=json.loads((OUTPUT/'preparation.json').read_text())
print('Typecheck SDK iPhone:',proof['native_typecheck_passed'])
print('Preflight PCM Mac:',proof['mac_pcm_pipeline_passed'])
for r in proof['checks']:print(r)
subprocess.run(['open',str(OUTPUT/'Benchmark.xcodeproj')],check=True)


## Pembanding CPU v1

Pilih Signing Team dan iPhone fisik pada project baru. Jalankan **CPU only**, lalu **Run selected track** jika membutuhkan pembanding CPU. GPU pada graph v1 telah diblokir; gunakan `encoder_glu_repair.ipynb` untuk pengujian berikutnya.

Waktu mencakup load/prediction/perpindahan model dan pooling; pemeriksaan integritas serta pembacaan fixture dilaporkan terpisah. Cache OS dapat memengaruhi hasil. Snapshot memori bukan peak memory. Project tetap terpisah dari aplikasi SongMap.


Untuk kelanjutan setelah crash MuQ 3000-frame, buka `encoder_glu_repair.ipynb`. Jangan menjalankan batch GPU dari project lama.
